In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

PATH_DRIVE = '/content/drive/MyDrive/Eafit/TDG/Codigo/Final/'
silver_layer = PATH_DRIVE + "capa_silver_final.csv"
gold_layer = PATH_DRIVE + "capa_gold_final.csv"
print(f"Ruta de dataset: {silver_layer}")
print(f"Ruta de dataset: {gold_layer}")

Ruta de dataset: /content/drive/MyDrive/Eafit/TDG/Codigo/Final/capa_silver_final.csv
Ruta de dataset: /content/drive/MyDrive/Eafit/TDG/Codigo/Final/capa_gold_final.csv


In [ ]:
import pandas as pd
import numpy as np
import re
df_silver = pd.read_csv(silver_layer, sep=';')

In [ ]:
def clean_numeric_value(val):
    # 1. Si ya es un número y no es nulo, lo devolvemos directamente
    if isinstance(val, (int, float)) and pd.notna(val):
        return float(val)

    # 2. Manejo de nulos y vacíos
    if pd.isna(val) or str(val).strip().lower() in ['none', 'nan', '']:
        return 0.0

    # 3. Convertir a string y limpiar
    s_val = str(val).lower().strip()

    if 'global' in s_val:
        return 2500.0

    # --- CAMBIO CRÍTICO AQUÍ ---
    # Eliminamos las comas de los miles (ej: 1,000.00 -> 1000.00)
    s_val = s_val.replace(',', '')

    # 4. Regex para detectar el número limpio
    match = re.search(r"(\d+\.?\d*)", s_val)
    if match:
        return float(match.group(1))

    return 0.0

# 2. Lista de columnas que NO deben tocarse
object_cols_to_keep = [
    'name', 'class_1', 'class_2', 'Role_1', 'Role_2',
    'Q_damage_type', 'W_damage_type', 'E_damage_type', 'R_damage_type',
    'champion', 'key_name' # Incluimos estas por si aún no las has dropeado
]

# 3. Aplicar la limpieza de forma segura
for col in df_silver.columns:
    if col not in object_cols_to_keep:
        # Usamos apply solo si hay strings; si ya es float64, lo dejamos tranquilo o lo validamos
        df_silver[col] = df_silver[col].apply(clean_numeric_value)

# 4. Verificación inmediata
print("--- VALIDACIÓN DE RANGOS ---")
print(df_silver[['name', 'Q_range', 'W_range']].head(5))


--- VALIDACIÓN DE RANGOS ---
      name  Q_range  W_range
0   Aatrox    625.0    825.0
1     Ahri    900.0    550.0
2    Akali    500.0      0.0
3   Akshan    850.0      0.0
4  Alistar    375.0    650.0


In [ ]:
df_silver.head(13)

,id,key_name,name,HP_13,AD_13,AR_13,MR_13,AS_13,Q_range,Q_hard_cc_dur,Q_soft_cc_dur,Q_base_dmg_13,Q_ratio_ad_13,Q_ratio_ap_13,Q_ratio_hp_13,Q_damage_type,Q_shield_base_13,W_range,W_hard_cc_dur,W_soft_cc_dur,W_base_dmg_13,W_ratio_ad_13,W_ratio_ap_13,W_ratio_hp_13,W_damage_type,W_shield_base_13,E_range,E_hard_cc_dur,E_soft_cc_dur,E_base_dmg_13,E_ratio_ad_13,E_ratio_ap_13,E_ratio_hp_13,E_damage_type,E_shield_base_13,R_range,R_hard_cc_dur,R_soft_cc_dur,R_base_dmg_13,R_ratio_ad_13,R_ratio_ap_13,R_ratio_hp_13,R_damage_type,R_shield_base_13,total_hard_cc_dur,total_soft_cc_dur,as_buff_passive,range_auto,champion,class_1,class_2,Role_1,Role_2
0,266.0,Aatrox,Aatrox,1898.30,114.75,90.56,54.45,0.829,625.0,1.00,0.0,446.25,5.7375,0.00,0.00,PHYSICAL_DAMAGE,0.0,825.0,0.00,1.5,60.0,0.8,0.00,0.0,PHYSICAL_DAMAGE,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,NONE,0.0,600.0,0.00,0.0,0.0,0.3,0.00,0.0,NONE,0.0,1.00,1.5,0.00,175.0,Aatrox,Juggernaut,NaN,Top,Jungle
1,103.0,Ahri,Ahri,1728.80,85.85,66.99,44.23,0.819,900.0,0.00,0.0,270.00,0.0000,1.00,0.00,MAGIC_DAMAGE,0.0,550.0,0.00,0.0,72.0,0.0,0.72,0.0,MAGIC_DAMAGE,0.0,1000.0,1.8,0.0,240.0,0.00,0.85,0.0,MAGIC_DAMAGE,0.0,500.0,0.00,0.0,125.0,0.0,0.35,0.0,MAGIC_DAMAGE,0.0,1.80,0.0,0.00,550.0,Ahri,Burst,NaN,Mid,NaN
2,84.0,Akali,Akali,1903.05,98.14,74.47,59.45,0.844,500.0,0.00,0.5,145.00,0.6500,0.60,0.00,MAGIC_DAMAGE,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,NONE,0.0,825.0,0.0,0.0,350.0,1.00,1.00,0.0,MAGIC_DAMAGE,0.0,675.0,0.00,0.0,420.0,0.0,0.90,0.0,MAGIC_DAMAGE,0.0,0.00,0.5,0.00,125.0,Akali,Assassin,NaN,Mid,Top
3,166.0,Akshan,Akshan,1781.65,84.85,77.47,44.23,0.813,850.0,0.00,0.0,330.00,1.0000,0.00,0.00,PHYSICAL_DAMAGE,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,NONE,0.0,800.0,0.0,0.0,250.0,0.25,0.00,0.0,PHYSICAL_DAMAGE,0.0,2500.0,0.00,0.0,630.0,2.7,0.00,0.0,PHYSICAL_DAMAGE,0.0,0.00,0.0,0.00,500.0,Akshan,Marksman,Assassin,Mid,NaN
4,12.0,Alistar,Alistar,1999.00,103.06,91.47,54.45,0.770,375.0,1.00,0.0,220.00,0.0000,0.80,0.00,MAGIC_DAMAGE,0.0,650.0,0.75,0.0,275.0,0.0,1.00,0.0,MAGIC_DAMAGE,0.0,125.0,1.0,0.0,80.0,0.00,0.70,0.0,MAGIC_DAMAGE,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,NONE,0.0,2.75,0.0,0.00,125.0,Alistar,Vanguard,NaN,Support,NaN
5,799.0,Ambessa,Ambessa,1834.50,95.85,88.66,54.45,0.796,650.0,0.00,0.0,270.00,1.0000,0.00,0.06,PHYSICAL_DAMAGE,0.0,325.0,0.00,0.0,125.0,0.0,0.00,0.0,PHYSICAL_DAMAGE,240.0,325.0,0.0,1.0,240.0,0.00,0.00,0.0,PHYSICAL_DAMAGE,0.0,1250.0,1.25,0.0,250.0,1.0,0.00,0.0,PHYSICAL_DAMAGE,0.0,1.25,1.0,0.00,125.0,Ambessa,Diver,Skirmisher,Top,Jungle
6,32.0,Amumu,Amumu,1714.30,98.61,76.80,54.45,0.888,1100.0,2.00,0.0,170.00,0.0000,0.85,0.00,MAGIC_DAMAGE,0.0,125.0,0.00,0.0,100.0,0.0,0.50,5.0,MAGIC_DAMAGE,0.0,350.0,0.0,0.0,185.0,0.00,0.50,0.0,MAGIC_DAMAGE,0.0,550.0,1.50,0.0,300.0,0.0,0.80,0.0,MAGIC_DAMAGE,0.0,3.50,0.0,0.00,125.0,Amumu,Vanguard,NaN,Jungle,Support
7,34.0,Anivia,Anivia,1557.40,86.04,70.28,44.23,0.773,1100.0,1.50,0.0,330.00,0.0000,0.70,0.00,MAGIC_DAMAGE,0.0,0.0,1.25,0.0,0.0,0.0,0.00,0.0,NONE,0.0,600.0,0.0,0.0,310.0,0.00,1.10,0.0,MAGIC_DAMAGE,0.0,750.0,0.00,1.0,320.0,0.0,0.82,0.0,MAGIC_DAMAGE,0.0,2.75,1.0,0.00,600.0,Anivia,Battlemage,NaN,Mid,Support
8,1.0,Annie,Annie,1611.20,79.02,66.80,44.23,0.703,625.0,1.75,0.0,260.00,0.0000,0.80,0.00,MAGIC_DAMAGE,0.0,600.0,0.00,0.0,230.0,0.0,0.80,0.0,MAGIC_DAMAGE,0.0,800.0,0.0,0.0,25.0,0.00,0.40,0.0,MAGIC_DAMAGE,60.0,600.0,0.00,0.0,275.0,0.0,0.75,0.0,MAGIC_DAMAGE,0.0,1.75,0.0,0.00,625.0,Annie,Burst,NaN,Mid,Support
9,523.0,Aphelios,Aphelios,1716.90,80.19,71.99,44.23,0.816,1450.0,1.00,2.0,0.00,3.0000,1.00,1.00,OTHER_DAMAGE,0.0,0.0,0.00,0.0,0.0,0.0,0.00,0.0,NONE,0.0,0.0,0.0,0.0,0.0,0.00,0.00,0.0,NONE,0.0,1300.0,1.35,0.0,175.0,1.1,0.00,0.0,PHYSICAL_DAMAGE,350.0,2.35,2.0,0.54,550.0,Aphelios,Marksman,NaN,Bot,NaN


In [ ]:
def build_gold_layer_lvl13_apex(df_silver):
    """
    Capa Gold Optimizada: Foco exclusivo en Nivel 13.
    Transforma los datos atómicos de Silver en métricas estratégicas para Apex Tier.
    """
    gold_rows = []
    LVL = 13  # Punto de referencia único

    # 1. Configuración de Pesos de Daño por Clase (Ajuste de balance)
    dmg_weights = {
        'Marksman': 1.2, 'Burst': 1.2, 'Artillery': 1.2, 'Battlemage': 1.2,
        'Assassin': 1.2, 'Skirmisher': 1.0, 'Diver': 1.0, 'Juggernaut': 1.0,
        'Vanguard': 0.8, 'Warden': 0.8, 'Enchanter': 0.8, 'Catcher': 1.0,
        'Specialist': 1.0
    }

    def clean(val):
        if pd.isna(val) or val == 'None' or val == '': return 0.0
        s_val = str(val).lower()
        if 'global' in s_val: return 3000.0  # Rango global estandarizado
        match = re.search(r"(\d+\.?\d*)", s_val)
        return float(match.group(1)) if match else 0.0

    for _, r in df_silver.iterrows():
        # --- IDENTIDAD Y CLASES ---
        entry = {
            'id': int(r.get('id')),
            'name': r.get('name'),
            'main_role': r.get('Role_1'),
            'sub_role': r.get('Role_2'),
            'class_1': r.get('class_1'),
            'class_2': r.get('class_2', '')
        }

        c1 = entry['class_1']
        c2 = entry['class_2'] if entry['class_2'] != 'None' else c1

        # Multiplicador de Daño 70/30 basado en Clase
        w1_dmg = dmg_weights.get(c1, 1.0)
        w2_dmg = dmg_weights.get(c2, w1_dmg)
        m_dmg = (w1_dmg * 0.7) + (w2_dmg * 0.3)

        # --- 1. DAÑO Y DPS ---
        ad_actual = r.get(f'AD_{LVL}', 0)
        phys_dmg = 0
        mag_dmg = 0

        for slot in ['Q', 'W', 'E', 'R']:
            base = r.get(f'{slot}_base_dmg_{LVL}', 0)
            ad_scale = ad_actual * r.get(f'{slot}_ratio_ad_{LVL}', 0)
            ap_scale = 100 * r.get(f'{slot}_ratio_ap_{LVL}', 0) # 100 AP base para proyección

            total_skill = (base + ad_scale + ap_scale) * m_dmg
            dtype = str(r.get(f'{slot}_damage_type', '')).upper()

            if 'PHYSICAL' in dtype: phys_dmg += total_skill
            elif 'MAGIC' in dtype: mag_dmg += total_skill

        entry['Gold_Phys_Dmg'] = round(phys_dmg, 2)
        entry['Gold_Mag_Dmg'] = round(mag_dmg, 2)

        # DPS: (AD * Velocidad de Ataque Real) * Multiplicador
        as_total = r.get(f'AS_{LVL}', 0) * (1 + r.get('as_buff_passive', 0))
        entry['Gold_DPS'] = round(ad_actual * as_total * m_dmg, 2)

        # --- 2. DURABILITY ---
        hp, ar, mr = r.get(f'HP_{LVL}', 0), r.get(f'AR_{LVL}', 0), r.get(f'MR_{LVL}', 0)
        # Fórmula de EHP (Effective Health Points)
        durability_base = (hp * (1 + (ar + mr) / 200)) / 1000
        # Bono de tanque (+20%)
        m_dur = 1.2 if any(c in [c1, c2] for c in ['Vanguard', 'Warden']) else 1.0
        entry['Gold_Durability'] = round(durability_base * m_dur, 2)

        # --- 3. CC POWER ---
        # Hard CC (2.5x peso) vs Soft CC (1.0x peso)
        cc_score = (r.get('total_soft_cc_dur', 0) * 1.0) + (r.get('total_hard_cc_dur', 0) * 2.5)
        m_cc = 1.2 if any(c in [c1, c2] for c in ['Catcher', 'Vanguard']) else 1.0
        entry['Gold_CC'] = round(cc_score * m_cc, 2)

        # 4. POKE Y ENGAGE (Lógica de Rescate Total) ---
        def rescue_rng(val):
            # Si es None, NaN, 0 o un string vacío, devolvemos 0.0 para el max()
            if pd.isna(val) or val == 0 or str(val).strip().lower() in ['none', '', '0', '0.0']:
                return 0.0
            try:
                return float(val)
            except:
                return 0.0

        # Limpiamos cada rango individualmente
        q = rescue_rng(r.get('Q_range'))
        w = rescue_rng(r.get('W_range'))
        e = rescue_rng(r.get('E_range'))
        # Removed the problematic line: r = rescue_rng(r.get('R_range'))
        auto = rescue_rng(r.get('range_auto'))


        # Buscamos el máximo. Si el máximo sigue siendo 0, usamos el autoataque
        max_rng = max([q, w, e])
        if max_rng == 0:
          max_rng = auto if auto > 0 else 125.0 # Valor mínimo para que no de 0

        entry['Gold_Poke'] = round(max_rng / 100, 2)

# --- 4. CÁLCULO DE ENGAGE PROFESIONAL ---
        # 1. Extraer rangos y CC (Asegurando tipos float)
        q_rng = float(r.get('Q_range', 0))
        w_rng = float(r.get('W_range', 0))
        e_rng = float(r.get('E_range', 0))
        r_rng = float(r.get('R_range', 0))

        hard_cc_total = float(r.get('total_hard_cc_dur', 0))

        # 2. Identificar clases para el Bonus
        c1 = str(r.get('class_1', '')).strip()
        c2 = str(r.get('Class_2', '')).strip() # Nota la C mayúscula de tu imagen

        # 3. Definir el Rango de Amenaza (Combo Máximo)
        # Buscamos qué habilidades básicas tienen CC para considerarlas como entrada
        q_has_cc = float(r.get('Q_hard_cc_dur', 0)) > 0
        w_has_cc = float(r.get('W_hard_cc_dur', 0)) > 0
        e_has_cc = float(r.get('E_hard_cc_dur', 0)) > 0

        # El alcance de iniciación es el máximo entre la R y cualquier básica con CC
        posibles_entradas = [r_rng]
        if q_has_cc: posibles_entradas.append(q_rng)
        if w_has_cc: posibles_entradas.append(w_rng)
        if e_has_cc: posibles_entradas.append(e_rng)

        max_threat_rng = max(posibles_entradas) if posibles_entradas else 0
        # Capamos a 2000 para no romper la escala con globales
        max_threat_rng = min(max_threat_rng, 2000.0)

        # 4. Aplicar Multiplicador de Rol (1.1x)
        # Si es Vanguard o Diver en cualquiera de sus dos clases, recibe el bono
        m_engage = 1.2 if any(role in [c1, c2] for role in ['Vanguard', 'Diver']) else 1.0

        # 5. Fórmula Final
        # (Rango / 1000) * (CC + 1) * Multiplicador
        score_base = (max_threat_rng / 1000) * (hard_cc_total + 1)
        entry['Gold_Engage'] = round(score_base * m_engage, 2)

# --- 5. CÁLCULO DE UTILITY (Escudos, Curación y Buffs) ---
        # 1. Sumamos escudos/curaciones de TODOS los slots (incluyendo E y R)
        # Usamos el sufijo _13 que vimos en tu imagen de Silver
        shield_q = float(r.get(f'Q_shield_base_{LVL}', 0))
        shield_w = float(r.get(f'W_shield_base_{LVL}', 0))
        shield_e = float(r.get(f'E_shield_base_{LVL}', 0))
        shield_r = float(r.get(f'R_shield_base_{LVL}', 0))

        # Base de utilidad: suma de escudos dividida por 100 para normalizar
        utility_base = (shield_q + shield_w + shield_e + shield_r) / 100

        # 2. Añadimos el valor del CC Suave (Slows/Silencios)
        # Los Enchanters aportan mucha utilidad vía Soft CC
        soft_cc_util = float(r.get('total_soft_cc_dur', 0)) * 0.1

        # 3. Añadimos el bono de Velocidad de Ataque pasiva (as_buff_pass)
        # Esto es utilidad pura para el ADC
        as_buff_util = float(r.get('as_buff_pass', 0)) * 1.0

        # 4. Multiplicador de Rol (Enchanter)
        # c1 y c2 ya están definidos en tu loop
        m_util = 2 if c1 == 'Enchanter' else (2 if c2 == 'Enchanter' else 1.0)

        # Score final de Utilidad
        total_utility = (utility_base + soft_cc_util + as_buff_util)
        entry['Gold_Utility'] = round(total_utility * m_util, 2)

        auto_range = r.get('range_auto', 125)
        # Ajuste de escalado para campeones específicos en Lvl 13
        if r['name'] == 'Tristana': auto_range += (8 * LVL)
        elif r['name'] == 'Senna': auto_range += 80 # Promedio de almas en Apex Lvl 13

        entry['Gold_Kiting'] = round(auto_range / 100, 2)

        gold_rows.append(entry)

    return pd.DataFrame(gold_rows)

In [ ]:
# Ejecución
df_gold = build_gold_layer_lvl13_apex(df_silver)

In [ ]:
df_gold.to_csv(gold_layer, index=False)